# TABLELLM Cross-Validation Evaluation

This notebook runs the TABLELLM-style evaluator for the lab explanation project. It can score saved model outputs quickly, or optionally load `RUCKBReasoning/TableLLM-8b` and generate predictions on held-out folds.

Use a GPU runtime if you want to run the TableLLM checkpoint directly.

## 1. Get the project into Colab

Upload this whole repository to Colab, clone it, or place it in Google Drive. Then set `PROJECT_DIR` below to the folder that contains `README.md`, `data/`, and `scripts/`.

In [ ]:
from pathlib import Path

# Option A: repository is directly in /content
PROJECT_DIR = Path('/content/NLP_medical_results_explanation')

# Option B: uncomment if your repo is in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = Path('/content/drive/MyDrive/NLP_medical_results_explanation')

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} does not exist. Upload/clone the repo first, or update PROJECT_DIR.'
    )

%cd {PROJECT_DIR}
!pwd
!ls

## 2. Install dependencies

For saved-output evaluation, this is light. For direct TableLLM inference, the same cell installs the Hugging Face and bitsandbytes stack used by the project.

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -r requirements-colab.txt

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

print('Dependencies installed. If Colab asks you to restart the runtime, restart and rerun from this cell.')

## 3. Configure evaluation

`PREDICTION_COLUMN` should be the column containing model outputs you want to evaluate. If you set it to `generated_text` while also using `generated_text` as the target, the scores will be perfect because this is only a pipeline sanity check.

In [ ]:
INPUT_CSV = 'data/lab_summaries_export.csv'
PROMPT_COLUMN = 'prompt'
TARGET_COLUMN = 'generated_text'
PREDICTION_COLUMN = 'generated_text'  # Change to fine_tuned_output, base_model_output, etc. when available.
OUTPUT_DIR = 'outputs/tablellm_cv_colab'
FOLDS = 5
MAX_ROWS = None  # Set to a small number like 20 for a quick smoke run.

df = pd.read_csv(INPUT_CSV)
print(df.shape)
print(df.columns.tolist())
df.head(2)

## 4. Evaluate saved outputs with cross-validation

This mode does not train a model. It assigns rows to folds, scores the selected prediction column against the target column, and writes visualizations.

In [ ]:
import subprocess

cmd = [
    'python', 'scripts/testing/evaluate_tablellm_cv.py',
    '--input', INPUT_CSV,
    '--prompt-column', PROMPT_COLUMN,
    '--target-column', TARGET_COLUMN,
    '--prediction-column', PREDICTION_COLUMN,
    '--folds', str(FOLDS),
    '--output-dir', OUTPUT_DIR,
]
if MAX_ROWS is not None:
    cmd += ['--max-rows', str(MAX_ROWS)]

print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 5. View the results

In [ ]:
summary_path = Path(OUTPUT_DIR) / 'tablellm_cv_summary.csv'
results_path = Path(OUTPUT_DIR) / 'tablellm_cv_results.csv'
fold_chart = Path(OUTPUT_DIR) / 'tablellm_cv_fold_scores.png'
operation_chart = Path(OUTPUT_DIR) / 'tablellm_cv_operation_scores.png'

summary = pd.read_csv(summary_path)
display(summary)

display(Image(filename=str(fold_chart)))
display(Image(filename=str(operation_chart)))

results = pd.read_csv(results_path)
display(results.head())

## 6. Optional: run the TableLLM checkpoint directly

This downloads `RUCKBReasoning/TableLLM-8b` from Hugging Face and runs generation on each held-out fold. Use a GPU runtime and start with `MAX_ROWS_FOR_MODEL = 5` or `10` before scaling up.

In [ ]:
RUN_TABLELLM_MODEL = False
MAX_ROWS_FOR_MODEL = 5
TABLELLM_OUTPUT_DIR = 'outputs/tablellm_cv_colab_model'

if RUN_TABLELLM_MODEL:
    from huggingface_hub import login
    login()  # Paste your Hugging Face token when prompted, if required.

    cmd = [
        'python', 'scripts/testing/evaluate_tablellm_cv.py',
        '--run-model',
        '--model-id', 'RUCKBReasoning/TableLLM-8b',
        '--load-4bit',
        '--input', INPUT_CSV,
        '--prompt-column', PROMPT_COLUMN,
        '--target-column', TARGET_COLUMN,
        '--folds', str(FOLDS),
        '--max-rows', str(MAX_ROWS_FOR_MODEL),
        '--output-dir', TABLELLM_OUTPUT_DIR,
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Set RUN_TABLELLM_MODEL = True to run direct TableLLM inference.')

## 7. Download outputs

In [ ]:
from google.colab import files

zip_path = 'tablellm_cv_outputs.zip'
!zip -qr {zip_path} {OUTPUT_DIR}
files.download(zip_path)